# "Squidward" Optimization

In [ ]:
import parametrization, sparse_matrices, mesh, numpy as np, importlib, pickle, wall_generation
from tri_mesh_viewer import TriMeshViewer
import vis, matplotlib
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

target_surface = mesh.Mesh("../examples/squidward_remesh.obj")
target_surface = target_surface.subdivide_loop(1)

# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(3, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(target_surface, parametrization.lscm(target_surface))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

print(lg.energy())
for i in range(1000): lg.runIteration()
print(lg.energy())

In [ ]:
for i in range(4000): lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin = alphaMin
lg.alphaMax = alphaMax
for i in range(4000): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surface, lg.uv())

In [ ]:
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegWeight):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegWeight
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 2000
    opts.gradTol = 1e-10
    parametrization.benchmark_reset()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)
    parametrization.benchmark_report()

In [ ]:
parametrization.benchmark_reset()
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 1e-1)
# with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 1e-3)
# with suppress_stdout(): optimize_rparam(rparam, 1e-6, 1e-6, 1e-3)
# with suppress_stdout(): optimize_rparam(rparam, 1e-7, 1e-7, 1e-4)
# with suppress_stdout(): optimize_rparam(rparam, 0.5e-7, 0.5e-7, 1e-4)
# with suppress_stdout(): optimize_rparam(rparam, 0.2e-7, 0.2e-7, 1e-4)
# with suppress_stdout(): optimize_rparam(rparam, 1e-7, 1e-7, 1e-4)
parametrization.benchmark_report()

In [ ]:
rparam.energy(rparam.EnergyType.Fitting)

In [ ]:
PET = parametrization.RegularizedParametrizerSVD.EnergyType
[rparam.energy(et) / reg for (et, reg) in [(PET.Fitting, 1.0), (PET.AlphaRegularization, rparam.alphaRegW), (PET.PhiRegularization, rparam.phiRegW), (PET.BendingRegularization, rparam.bendRegW)]]

In [ ]:
PET = parametrization.RegularizedParametrizerSVD.EnergyType
[rparam.energy(et) / reg for (et, reg) in [(PET.Fitting, 1.0), (PET.AlphaRegularization, rparam.alphaRegW), (PET.PhiRegularization, rparam.phiRegW), (PET.BendingRegularization, rparam.bendRegW)]]

In [ ]:
visualization.singularValueHistogram(parametrization.RegularizedParametrizerSVD(lg))

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
visualization.visualize(rparam)

In [ ]:
visualization.visualize(lg)

In [ ]:
kappa_idx = 0
tri_kappa = np.array([rparam.curvature3d(i, kappa_idx)[0] for i in range(target_surface.numTris())])
tri_d = np.array([rparam.curvature3d(i, kappa_idx)[1] for i in range(target_surface.numTris())])

In [ ]:
vectors = rparam.tubeDirections()
#vectors = ci.d_1
#vectors = tri_d
vf = vis.fields.VectorField(vectors, glyph=vis.fields.VectorGlyph.CYLINDER, colormap=matplotlib.cm.viridis, align=vis.fields.VectorAlignment.CENTER)
sf = vis.fields.ScalarField(tri_kappa, colormap=matplotlib.cm.viridis, domainType=vis.fields.DomainType.PER_TRI)
#vf = None
viewer = TriMeshViewer(target_surface, width=1400, height=640, vectorField=vf, scalarField=sf)
viewer.arrowSize = 40
#viewer.showWireframe()
viewer.setCameraParams(((0.5860034920267359, -3.035291050193042, 0.6228616949298365),
 (-0.0173304352854086, 0.39865202259301563, 0.9169385044239974),
 (-0.07872449082034016, -0.22380850423877954, -0.6120338005211532)))
viewer.show()

In [ ]:
np.savetxt('results/squidward/param.txt', rparam.uv())

In [ ]:
np.savetxt('results/squidward/param_hres.txt', rparam.uv())

In [ ]:
np.savetxt('results/squidward/param_hres.txt', rparam.uv())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surface, np.loadtxt('results/squidward/param_hres.txt'))

In [ ]:
alphas = rparam.getAlphas()

In [ ]:
widths = wwf.wallWidthForCanonicalWidth(wwf.canonicalWallWidthForStretchFactor(np.clip(rparam.getAlphas(), alphaMin, alphaMax)), 10)
(np.min(widths), np.max(widths))

## Upsampling and channel generation

In [ ]:
nsubdiv=3
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)

In [ ]:
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=3.75)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.5,
                                              minContourLen=0.75)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)

In [ ]:
import write_line_mesh
write_line_mesh.write_line_mesh('squidward.obj', pts, edges)

## Meshing and inflation simulation

In [ ]:
import parametrization, sparse_matrices, mesh, numpy as np, importlib, pickle, wall_generation
from tri_mesh_viewer import TriMeshViewer
import vis, matplotlib
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

In [ ]:
target_surface = mesh.Mesh("../examples/squidward_remesh.obj")
target_surface = target_surface.subdivide_loop(1)

In [ ]:
m, fuseMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, 0.15)
m.save('results/squidward/top_sheet_f3.75_laptop.msh')
np.savetxt('results/squidward/fuse_markers_f3.75_laptop.txt', fuseMarkers)
# m = mesh.Mesh('results/squidward/top_sheet_f3.75.msh')
# fuseMarkers = np.loadtxt('results/squidward/fuse_markers_f3.75.txt')
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
m.save('test.msh')

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)
bv = isheet.mesh().boundaryVertices()
bdryVars = [isheet.varIdx(0, i, c) for i in bv for c in range(3)]

# uv = np.loadtxt('results/squidward/param_hres.txt')
uv = rparam.uv()
paramSampler = mesh.SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surface.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surface.vertices())

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
harmonicPositions = parametrization.harmonic(isheet.mesh(), liftedSheetPositions[bv])
harmonicPositions += 0.15 * (liftedSheetPositions - harmonicPositions)

In [ ]:
isheet.setUninflatedDeformation(harmonicPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
import tri_mesh_viewer
importlib.reload(tri_mesh_viewer)
viewer = tri_mesh_viewer.TriMeshViewer(isheet.visualizationMesh(), width=1024, height=640)
#viewer.showWireframe()
viewer.setCameraParams(((2.855530256940937, 1.002991214638454, -1.1322350499425347),
 (-0.3303947111142514, -0.32128434449313503, -0.8874771573687668),
 (0.04746981475129009, 0.1255260177204086, 0.23082442618238475)))
viewer.show()

In [ ]:
viewer.showWireframe(False)

In [ ]:
viewer.update(mesh=target_surface, preserveExisting=True)

In [ ]:
viewer.update(mesh=isheet.visualizationMesh(), preserveExisting=False)

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-6
opts.niter = iterations_per_output

In [ ]:
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surface)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [ ]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True

In [ ]:
targetAttractedSheet.fittingWeight = 1e-8

In [ ]:
isheet.setIdentityDeformation()

In [ ]:
mkdir results/squidward/inflation

In [ ]:
import time
inflation.benchmark_reset()
isheet.setUseTensionFieldEnergy(False)
isheet.setUseHessianProjectedEnergy(True)
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
# isheet.pressure = 1.0
isheet.pressure = 0.25
fixedVars = bdryVars
#fixedVars = isheet.rigidMotionPinVars
for step in range(int(niter / iterations_per_output)):
    # cr = inflation.inflation_newton(targetAttractedSheet, [], opts)
    cr = inflation.inflation_newton(isheet, fixedVars, opts)
    # isheet.writeDebugMesh('results/squidward/inflation/step_{}.msh'.format(step))
    viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.01) # Allow some mesh synchronization time for pythreejs
    if cr.numIters() < iterations_per_output: break
inflation.benchmark_report()

In [ ]:
import fd_validation
import importlib
importlib.reload(fd_validation);

In [ ]:
isheet.setUseTensionFieldEnergy(True)

In [ ]:
isheet.setUseTensionFieldEnergy(False)

In [ ]:
perturb = np.random.uniform(-1.0, 1.0, isheet.numVars())

In [ ]:
eps, errors, an_full = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Full, perturb=perturb, fixedVars=bdryVars)
from matplotlib import pyplot as plt
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors, an_full = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Elastic, perturb=perturb, fixedVars=bdryVars)
from matplotlib import pyplot as plt
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors, an_full = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Pressure, perturb=perturb, fixedVars=bdryVars)
from matplotlib import pyplot as plt
plt.loglog(eps, errors)
plt.show()

In [ ]:
np.dot(isheet.gradient(energyType=isheet.EnergyType.Elastic), perturb)

In [ ]:
np.dot(isheet.gradient(energyType=isheet.EnergyType.Full), perturb)

In [ ]:
np.dot(isheet.gradient(energyType=isheet.EnergyType.Pressure), perturb)

In [ ]:
eps, errors, an_full = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType., perturb=perturb)
from matplotlib import pyplot as plt
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Full, perturb=perturb)
from matplotlib import pyplot as plt
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Elastic, perturb=perturb)
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Pressure, perturb=perturb)
plt.loglog(eps, errors)
plt.show()

In [ ]:
eps, errors = fd_validation.gradConvergence(isheet, energyType=isheet.EnergyType.Elastic, perturb=perturb)
plt.loglog(eps, errors)
plt.show()

In [ ]:
rso = inflation.ReducedSheetOptimizer(targetAttractedSheet, opts)

In [ ]:
rso.energy()

In [ ]:
import fd_validation
# rso.commitDesign()
inflation.benchmark_reset()
print(fd_validation.validateGrad(rso, fd_eps=1e-6, etype=rso.EnergyType.Fitting))
inflation.benchmark_report()

In [ ]:
import fd_validation
fd_validation.validateGrad(isheet, fd_eps=1e-7, etype=isheet.EnergyType.Elastic)

TODO: Avoid cancellation errors in the total potential energy gradient. Either use a higher precision in evaluating the gradient (boost multiprecision/mpfr)
or try decreasing the pressure so that the magnitude of each gradient term is less? Or make the gradient tolerance relative to the individual terms of the gradient (Elastic, Pressure, Fitting...).
We should study how these compare to the initial gradient--presumably the individual terms' magnitudes are similar to the initial gradient due to the pressure remaining constant.

In [ ]:
isheet.gradient(energyType=isheet.EnergyType.Elastic)

In [ ]:
isheet.gradient(energyType=isheet.EnergyType.Pressure)

In [ ]:
fd_validation.validateGrad(targetAttractedSheet, fd_eps=1e-7, etype=targetAttractedSheet.EnergyType.Fitting)

In [ ]:
fd_validation.validateGrad(targetAttractedSheet, fd_eps=1e-7, etype=targetAttractedSheet.EnergyType.Full)

In [ ]:
fd_validation.validateGrad(targetAttractedSheet, fd_eps=1e-6, etype=targetAttractedSheet.EnergyType.Simulation)

In [ ]:
fd_validation.validateGrad(targetAttractedSheet, fd_eps=1e-7)

TODO: figure out why the accuracy of the finite differences of the fitting gradient degrades so poorly--the surface is smooth so switching between triangles shouldn't give that bad a discontinuity.
(Find and visualize the problematic vertex)

In [ ]:
fd_validation.validateHessian(targetAttractedSheet, fd_eps=0.99e-7, etype=targetAttractedSheet.EnergyType.Fitting)

In [ ]:
H = targetAttractedSheet.hessian()

In [ ]:
d = H.solve(-targetAttractedSheet.gradient())

In [ ]:
np.dot(d, targetAttractedSheet.gradient())

In [ ]:
xorig = targetAttractedSheet.getVars()

In [ ]:
d = -targetAttractedSheet.gradient()

In [ ]:
def energyAt(x):
    targetAttractedSheet.setVars(x)
    e, esim, efit = targetAttractedSheet.energy(), targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Simulation), targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)
    return (e, esim, efit)
stepLengths = np.linspace(0, 0.1, 200)
energies = [energyAt(xorig + sl * d) for sl in stepLengths]
targetAttractedSheet.setVars(xorig)

In [ ]:
gfit = targetAttractedSheet.gradient(targetAttractedSheet.EnergyType.Fitting)
gsim = targetAttractedSheet.gradient(targetAttractedSheet.EnergyType.Simulation)
g    = targetAttractedSheet.gradient()

In [ ]:
targetAttractedSheet.gradient(targetAttractedSheet.EnergyType.Simulation)

In [ ]:
np.argmax(np.abs(targetAttractedSheet.gradient()))

In [ ]:
energiesSteepestDescent = np.array(energies)

In [ ]:
energiesNewtonStep = np.array(energies)

In [ ]:
norm(d)

In [ ]:
norm(d) / norm(targetAttractedSheet.gradient())

In [ ]:
targetAttractedSheet.energy()

In [ ]:
targetAttractedSheet.setVars(xorig - 0.05 * g)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
# plt.plot(stepLengths, energies[:, 0], stepLengths, energies[:, 1], stepLengths, energies[:, 2])
# plt.plot(stepLengths, energiesNewtonStep[:, 0], stepLengths, energiesSteepestDescent[:, 0])
plt.plot(stepLengths, energiesSteepestDescent[:, 0])
plt.show()

In [ ]:
energiesSteepestDescent[-1, 0]

In [ ]:
energiesSteepestDescent[0, 0]

In [ ]:
isheet.setUseTensionFieldEnergy(True)

In [ ]:
fd_validation.validateHessian(isheet, fd_eps=1e-8, etype=isheet.EnergyType.Elastic)

In [ ]:
isheet.tensionStateHistogram()